# 01 - Normalized Pipeline Walkthrough (CSV-Only)

This notebook is intentionally **database-free**. It walks through the normalization ideas from this commit using only CSVs in `seeder_data/data`.

## What This Shows

- Canonical fighter identity linking Tapology and UFCStats IDs
- Event + bout stitching from source CSVs
- Sport/weight-class normalization logic (division key + weight limit/unit)
- Promotion strength usage from `promotions_with_sports.csv` (single strength concept)

All outputs are printed or displayed as pandas DataFrames.

In [ ]:
import re
from pathlib import Path

import pandas as pd

DATA_DIR = Path('seeder_data/data')

required_files = [
    'tapology_fighters.csv',
    'tapology_bout_fighters.csv',
    'tapology_bouts.csv',
    'tapology_events.csv',
    'ufcstats_fighters.csv',
    'ufcstats_fights.csv',
    'ufcstats_events.csv',
    'fighter_id_map.csv',
    'bout_id_map.csv',
    'promotions_with_sports.csv',
]

for f in required_files:
    p = DATA_DIR / f
    if not p.exists():
        raise FileNotFoundError(f'Missing required CSV: {p}')

print('Using data directory:', DATA_DIR.resolve())

In [ ]:
frames = {
    'tapology_fighters': pd.read_csv(
        DATA_DIR / 'tapology_fighters.csv', dtype=str, keep_default_na=False, na_filter=False
    ),
    'tapology_bout_fighters': pd.read_csv(
        DATA_DIR / 'tapology_bout_fighters.csv', dtype=str, keep_default_na=False, na_filter=False
    ),
    'tapology_bouts': pd.read_csv(DATA_DIR / 'tapology_bouts.csv', dtype=str, keep_default_na=False, na_filter=False),
    'tapology_events': pd.read_csv(DATA_DIR / 'tapology_events.csv', dtype=str, keep_default_na=False, na_filter=False),
    'ufcstats_fighters': pd.read_csv(
        DATA_DIR / 'ufcstats_fighters.csv', dtype=str, keep_default_na=False, na_filter=False
    ),
    'ufcstats_fights': pd.read_csv(DATA_DIR / 'ufcstats_fights.csv', dtype=str, keep_default_na=False, na_filter=False),
    'ufcstats_events': pd.read_csv(DATA_DIR / 'ufcstats_events.csv', dtype=str, keep_default_na=False, na_filter=False),
    'fighter_id_map': pd.read_csv(DATA_DIR / 'fighter_id_map.csv', dtype=str, keep_default_na=False, na_filter=False),
    'bout_id_map': pd.read_csv(DATA_DIR / 'bout_id_map.csv', dtype=str, keep_default_na=False, na_filter=False),
    'promotions_with_sports': pd.read_csv(
        DATA_DIR / 'promotions_with_sports.csv', dtype=str, keep_default_na=False, na_filter=False
    ),
}

pd.DataFrame(
    [(k, len(v), list(v.columns)[:6]) for k, v in frames.items()], columns=['csv', 'rows', 'first_columns']
).sort_values('rows', ascending=False)

## 1) Pick Example Fighter

In [ ]:
example_slug = 'mirko-filipovic-cro-cop'

tap_f = frames['tapology_fighters']
map_f = frames['fighter_id_map']
ufc_f = frames['ufcstats_fighters']

fighter_row = tap_f.loc[tap_f['tapology_fighter_id'] == example_slug, ['tapology_fighter_id', 'name', 'country', 'dob']]
if fighter_row.empty:
    raise ValueError(f'Fighter slug not found: {example_slug}')

id_map_row = map_f.loc[map_f['tapology_fighter_id'] == example_slug]
if id_map_row.empty:
    raise ValueError(f'No UFCStats mapping found for: {example_slug}')

ufc_id = id_map_row.iloc[0]['ufcstats_fighter_id']
ufc_profile = ufc_f.loc[
    ufc_f['ufcstats_fighter_id'] == ufc_id, ['ufcstats_fighter_id', 'name', 'record', 'dob', 'stance']
]

print('Selected tapology slug:', example_slug)
print('Mapped ufcstats id:', ufc_id)

fighter_row

In [ ]:
print('Fighter ID map row:')
print(id_map_row.to_string(index=False))

print('UFCStats profile row:')
ufc_profile

## 2) Build Tapology Timeline Rows

In [ ]:
tap_bf = frames['tapology_bout_fighters']
tap_b = frames['tapology_bouts']

fighter_tap_rows = tap_bf[tap_bf['tapology_fighter_id'] == example_slug].copy()

fighter_tap_timeline = fighter_tap_rows.merge(tap_b, on='tapology_bout_id', how='left', suffixes=('_fighter', '_bout'))

fighter_tap_timeline['event_date'] = pd.to_datetime(fighter_tap_timeline['event_date'], errors='coerce')

tap_view_cols = [
    'event_date',
    'sport',
    'event_name',
    'promotion_name',
    'result',
    'opponent_name',
    'weight_class',
    'weight_lbs',
    'method',
    'method_details',
    'tapology_bout_id',
]

fighter_tap_timeline = fighter_tap_timeline.sort_values('event_date')[tap_view_cols]
print('Tapology timeline rows:', len(fighter_tap_timeline))
fighter_tap_timeline.head(20)

## 3) Build UFCStats Timeline Rows

In [ ]:
uf_fights = frames['ufcstats_fights'].copy()
uf_events = frames['ufcstats_events'].copy()
uf_fighters = frames['ufcstats_fighters'][['ufcstats_fighter_id', 'name']].rename(columns={'name': 'fighter_name'})
bout_map = frames['bout_id_map'][['ufcstats_fight_id', 'tapology_bout_id']]

uf_fighter_rows = uf_fights[(uf_fights['red_fighter_id'] == ufc_id) | (uf_fights['blue_fighter_id'] == ufc_id)].copy()

uf_fighter_rows = uf_fighter_rows.merge(
    uf_events[['ufcstats_event_id', 'event_name', 'event_date']], on='ufcstats_event_id', how='left'
)
uf_fighter_rows = uf_fighter_rows.merge(bout_map, on='ufcstats_fight_id', how='left')

# derive opponent id/name
uf_fighter_rows['opponent_id'] = uf_fighter_rows.apply(
    lambda r: r['blue_fighter_id'] if r['red_fighter_id'] == ufc_id else r['red_fighter_id'], axis=1
)
uf_fighter_rows = uf_fighter_rows.merge(
    uf_fighters, left_on='opponent_id', right_on='ufcstats_fighter_id', how='left'
).rename(columns={'fighter_name': 'opponent_name'})

uf_fighter_rows['event_date'] = pd.to_datetime(uf_fighter_rows['event_date'], errors='coerce')
uf_fighter_rows['result'] = uf_fighter_rows['winner_fighter_id'].map(
    lambda x: 'W' if x == ufc_id else ('D' if x == '' else 'L')
)

uf_view = uf_fighter_rows[
    [
        'event_date',
        'event_name',
        'result',
        'opponent_name',
        'weight_class',
        'gender',
        'method',
        'method_detail',
        'round',
        'time',
        'ufcstats_fight_id',
        'tapology_bout_id',
    ]
].sort_values('event_date')

print('UFCStats timeline rows:', len(uf_view))
uf_view.head(20)

## 4) Unified Cross-Source Timeline

In [ ]:
tap_unified = fighter_tap_timeline.assign(source='tapology').rename(
    columns={'method': 'method_raw', 'method_details': 'method_detail_raw'}
)

uf_unified = uf_view.assign(source='ufcstats', sport='mma', promotion_name='UFCStats', weight_lbs='').rename(
    columns={'method': 'method_raw', 'method_detail': 'method_detail_raw'}
)

common_cols = [
    'event_date',
    'source',
    'sport',
    'event_name',
    'promotion_name',
    'result',
    'opponent_name',
    'weight_class',
    'weight_lbs',
    'method_raw',
    'method_detail_raw',
]

timeline = pd.concat([tap_unified[common_cols], uf_unified[common_cols]], ignore_index=True).sort_values(
    ['event_date', 'source']
)

print('Unified rows:', len(timeline))
timeline.head(30)

## 5) CSV-Only Weight Normalization Demo

In [ ]:
def parse_decimal(s: str):
    try:
        return float(str(s).strip())
    except Exception:
        return None


KG_SPORTS = {'kickboxing', 'shootboxing', 'lethwei'}
KG_HEURISTIC_MAX = 120
MMA_CLASSIC_LIMITS = {115, 125, 135, 145, 155, 170, 185, 205, 265}
KG_TO_LB = 2.20462


def infer_weight_unit_and_value(weight_class_raw: str, weight_lbs_raw: str, sport_key: str):
    txt = (weight_class_raw or '').lower().strip()
    numeric = None
    m = re.search(r'(\d+(?:\.\d+)?)', txt)
    if m:
        numeric = float(m.group(1))

    weight_lbs_col = parse_decimal(weight_lbs_raw)

    unit = None
    value = None
    value_lbs = None

    if 'kg' in txt:
        if numeric is not None:
            unit = 'kg'
            value = numeric
            value_lbs = round(numeric * KG_TO_LB, 2)
    elif 'lb' in txt or 'lbs' in txt:
        if numeric is not None:
            unit = 'lb'
            value = numeric
            value_lbs = numeric
    elif numeric is not None:
        unit = 'lb'
        value = numeric
        value_lbs = numeric
        if sport_key in KG_SPORTS and numeric <= KG_HEURISTIC_MAX and numeric not in MMA_CLASSIC_LIMITS:
            unit = 'kg'
            value_lbs = round(numeric * KG_TO_LB, 2)
    elif weight_lbs_col is not None:
        unit = 'lb'
        value = weight_lbs_col
        value_lbs = weight_lbs_col

    return unit, value, value_lbs


MMA_PHRASE_TO_DIV = {
    'atomweight': 'MMA_ATOM',
    'strawweight': 'MMA_STRAW',
    'flyweight': 'MMA_FLY',
    'bantamweight': 'MMA_BW',
    'featherweight': 'MMA_FW',
    'lightweight': 'MMA_LW',
    'welterweight': 'MMA_WW',
    'middleweight': 'MMA_MW',
    'light heavyweight': 'MMA_LHW',
    'heavyweight': 'MMA_HW',
    'open': 'MMA_OPEN',
}


def infer_division_key(weight_class_raw: str, sport_key: str):
    txt = (weight_class_raw or '').lower()
    if sport_key == 'mma':
        for phrase, key in MMA_PHRASE_TO_DIV.items():
            if phrase in txt:
                return key
        return 'MMA_UNKNOWN'

    if 'open' in txt:
        return f'{sport_key.upper()}_OPEN'
    if 'catch' in txt:
        return f'{sport_key.upper()}_CATCHWEIGHT'
    if not txt.strip():
        return f'{sport_key.upper()}_UNKNOWN'
    cleaned = re.sub(r'[^A-Z0-9]+', '_', txt.upper()).strip('_')
    return f'{sport_key.upper()}_{cleaned}'


norm = timeline.copy()
norm[['weight_unit', 'weight_value', 'weight_lbs_norm']] = norm.apply(
    lambda r: pd.Series(infer_weight_unit_and_value(r['weight_class'], r['weight_lbs'], r['sport'])), axis=1
)
norm['division_key'] = norm.apply(lambda r: infer_division_key(r['weight_class'], r['sport']), axis=1)
norm['is_catchweight'] = norm['weight_class'].str.contains('catch', case=False, na=False)
norm['is_openweight'] = norm['weight_class'].str.contains('open', case=False, na=False)

norm[
    [
        'event_date',
        'source',
        'sport',
        'weight_class',
        'weight_unit',
        'weight_value',
        'weight_lbs_norm',
        'division_key',
        'is_catchweight',
        'is_openweight',
    ]
].head(40)

## 6) Promotion Strength Input (Single Strength Field)

In [ ]:
promo = frames['promotions_with_sports'][['tapology_promotion_id', 'name', 'sport_type', 'baseline_strength']].copy()
promo = promo.rename(columns={'baseline_strength': 'source_strength'})

# attach to tapology timeline by tapology promotion id through tapology_bouts
tap_bouts = frames['tapology_bouts'][['tapology_bout_id', 'promotion_tapology_id', 'promotion_name']].copy()
tap_with_promo = (
    frames['tapology_bout_fighters'][frames['tapology_bout_fighters']['tapology_fighter_id'] == example_slug]
    .merge(tap_bouts, on='tapology_bout_id', how='left')
    .merge(promo, left_on='promotion_tapology_id', right_on='tapology_promotion_id', how='left')
)

promotion_col = 'promotion_name_x' if 'promotion_name_x' in tap_with_promo.columns else 'promotion_name'

print('Rows with promotion strength attached:', len(tap_with_promo))
tap_with_promo[['tapology_bout_id', promotion_col, 'name', 'sport_type', 'source_strength']].head(25)

## 7) Quick Sanity Outputs

In [ ]:
print('Timeline rows by source:')
print(timeline['source'].value_counts())

print('\nTimeline rows by sport:')
print(timeline['sport'].value_counts().head(15))

print('\nUnique normalized division keys (sample):')
print(sorted(norm['division_key'].dropna().unique())[:25])

print('\nWeight unit counts:')
print(norm['weight_unit'].value_counts(dropna=False))

## Notes

- This notebook intentionally does not touch the database.
- It demonstrates the same normalization concepts using only source CSVs and pandas transformations.
- If you want, we can add a second notebook that compares this CSV-only view to canonical DB rows as a validation audit.